In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.structured-OCR.read-business-card",
        description="Extract relevent information from a business card",
        worker_release="qwen3-instruct",
        text_prompt="""
        You are given an image that may contain one or more business cards.
        Your task is to extract structured data from each business card clearly visible in the image.

        Return ONLY valid CSV text.
        Do not include explanation.
        Do not include markdown formatting (e.g., do not use ```csv).
        Do not include commentary.
        ---------------------------------------
        INSTRUCTIONS:

        1. Extract only text that is clearly readable.
        2. Do NOT guess or infer missing values. If a field is not visible, leave it blank (an empty space between commas).
        3. Every single row MUST contain exactly 7 commas, representing 8 columns, even if fields are empty.
        4. STRICT COLUMN MATCHING: You must place the correct type of data into the correct column. Do NOT shift data to the left if a previous field is missing. For example, a phone number must ALWAYS go in the Phone column, never in the Email column.
        5. If there are multiple business cards, return a new data row for each individual card.
        6. Preserve capitalization exactly as shown.
        7. If a field contains a comma (Company name), you MUST enclose that entire field in double quotes.
        8. If the image does not contain any business cards, return ONLY the header row.

        ---------------------------------------
        RETURN THIS EXACT CSV HEADER STRUCTURE AS THE FIRST LINE:

        First Name,Last Name,Title,Company,Email,Phone,Website

        Return only the CSV text.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=250,
            image_size=1000
        ),
        is_public=False
    )
]

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.structured-OCR.read-business-card:latest"
   )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.png")
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")

### Evaluation Flow

In [ ]:
from pathlib import Path

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.structured-OCR.read-business-card:latest"
    )
])

all_results = {}

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)
    directory_path = Path("/content/sample_data")
    for item in directory_path.iterdir():
        job = endpoint.upload(str(item))
        file_results = []

        while result := job.predict():
            file_results.append(result)

        all_results[item.name] = file_results

output_path = Path("/content/sample_data/output.json")
with open(output_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("Done")